## 4. Make the missing bridge explicit

We *declare* a one-edge translation: a successful Alpha diagnostic
`separable=true` for `a,b` becomes a rational edge difference 1 on a
newly declared Symbol Game board. Source digest, a freshly rerun Alpha
execution, local replay summary, and the mapping are gate inputs. A
submitted success string alone is refused. There is no claim
that a profile distinction mathematically equals an edge difference.
The Paper 032 governor independently recomputes its own finite board.


In [ ]:
import json
import os
import shutil
import subprocess
import sys
import tempfile
from fractions import Fraction
from hashlib import sha256
from pathlib import Path


def find_repositories():
    candidates = [Path.cwd(), *Path.cwd().parents]
    extra = os.environ.get("GENESIS_REPOS_HOME")
    if extra:
        candidates.insert(0, Path(extra))
    for root in candidates:
        has_genesis = (root / "genesis" / "3_alpha_mathematical_language" / "Cargo.toml").is_file() or (root / "3_alpha_mathematical_language" / "Cargo.toml").is_file()
        has_symbol = (root / "the_symbol_game" / "omega_engine_education").is_dir() or (root / "omega_engine_education").is_dir()
        if has_genesis and has_symbol:
            g_root = root / "genesis" if (root / "genesis").is_dir() else root
            s_root = root / "the_symbol_game" if (root / "the_symbol_game").is_dir() else root
            return g_root, s_root
    raise RuntimeError(
        "Place genesis and the_symbol_game beside each other or set GENESIS_REPOS_HOME"
    )


GENESIS, SYMBOL = find_repositories()
REPOS_HOME = GENESIS.parent
ALPHA = Path(os.environ.get("GENESIS_ALPHA_ROOT", GENESIS / "3_alpha_mathematical_language"))
_PFP_PAPERS = (
    SYMBOL
    / "omega_engine_atlas"
    / "coda_atlas"
    / "F004_papers"
    / "015_pfp"
)
PAPER33 = _PFP_PAPERS / "005_pfp_practitioners_field_manual" / "code"
PAPER32 = _PFP_PAPERS / "004_pfp_native_verification_and_evidence_custody" / "code"
import importlib.util


def _load(name, path):
    spec = importlib.util.spec_from_file_location(name, path)
    module = importlib.util.module_from_spec(spec)
    sys.modules[name] = module
    spec.loader.exec_module(module)
    return module


_governor = _load("l1_symbol_game_governor", PAPER32 / "l1_symbol_game_governor.py")
ConstraintGraph = _governor.ConstraintGraph
Edge = _governor.Edge
L1SymbolGameGovernor = _governor.L1SymbolGameGovernor


def digest(path):
    return sha256(Path(path).read_bytes()).hexdigest()


source_id = {
    "alpha_front_end_sha256": digest(ALPHA / "src" / "lib.rs"),
    "paper032_governor_sha256": digest(PAPER32 / "l1_symbol_game_governor.py"),
    "paper033_toolkit_sha256": digest(PAPER33 / "symbol_game_toolkit.py"),
}
print(json.dumps(source_id, indent=2))


In [ ]:
cargo = shutil.which("cargo")
if cargo is None:
    raise RuntimeError("Rust/Cargo is required for this source-backed lab")
TARGET = Path(tempfile.gettempdir()) / "genesis_alpha_omega_practicum_target"
build_env = dict(os.environ, CARGO_TARGET_DIR=str(TARGET))
build = subprocess.run(
    [cargo, "build", "--offline", "--quiet", "--bin", "alpha"],
    cwd=ALPHA,
    env=build_env,
    capture_output=True,
    text=True,
    timeout=180,
)
if build.returncode:
    raise RuntimeError("Alpha build refused: " + build.stderr)
binary = TARGET / "debug" / ("alpha.exe" if os.name == "nt" else "alpha")


def run_alpha(source):
    with tempfile.TemporaryDirectory(prefix="alpha-practicum-") as tmp:
        path = Path(tmp) / "lesson.alpha"
        path.write_text(source, encoding="utf-8")
        result = subprocess.run(
            [str(binary), str(path)], cwd=ALPHA, capture_output=True, text=True, timeout=30
        )
    return {
        "source_sha256": sha256(source.encode("utf-8")).hexdigest(),
        "binary_sha256": digest(binary),
        "exit": result.returncode,
        "stdout": result.stdout,
        "stderr": result.stderr,
    }


first_source = (ALPHA / "examples" / "first.alpha").read_text(encoding="utf-8")
first = run_alpha(first_source)
assert first["exit"] == 0, first["stderr"]
assert "replay=checked" in first["stdout"]
print(first["stdout"])

In [ ]:
class AdapterRefusal(ValueError):
    pass


declared_adapter = {
    "alpha_source_sha256": first["source_sha256"],
    "alpha_diagnostic": "separable=true",
    "vertices": ("a", "b"),
    "edge": ("a", "b", 1),
    "declared_difference": Fraction(1),
    "interpretation": "teaching convention only: separation maps to edge value +1",
}


def consume(receipt, adapter, source_text=first_source):
    if not adapter or receipt["exit"] != 0 or "replay=checked" not in receipt["stdout"]:
        raise AdapterRefusal("missing successful local Alpha replay")
    if sha256(source_text.encode("utf-8")).hexdigest() != adapter["alpha_source_sha256"]:
        raise AdapterRefusal("source identity changed")
    if receipt["source_sha256"] != adapter["alpha_source_sha256"]:
        raise AdapterRefusal("source identity changed")
    fresh = run_alpha(source_text)
    if receipt != fresh:
        raise AdapterRefusal("Alpha replay receipt mismatch")
    if adapter["alpha_diagnostic"] not in receipt["stdout"]:
        raise AdapterRefusal("required observation absent")
    if not adapter["interpretation"]:
        raise AdapterRefusal("no declared interpretation")
    u, v, weight = adapter["edge"]
    graph = ConstraintGraph(vertices=adapter["vertices"], edges=(Edge(u, v, weight),))
    governor = L1SymbolGameGovernor(graph)
    assertion = governor.step_assert(u, v, adapter["declared_difference"])
    if not assertion.admitted:
        raise AdapterRefusal("governor refused edge: " + assertion.status)
    verify = governor.step_verify("ZERO_OBSTRUCTION")
    return governor, assertion, verify


host, assertion, verification = consume(first, declared_adapter)
assert assertion.admitted and verification.admitted
assert host.primary_obstruction() == Fraction(0)
print("host:", assertion.status, verification.status, "Phi_1=", host.primary_obstruction())

**Assessment C (5 points).** Each gate should have a failing mutation.
A changed Alpha source, forged output, absent mapping, and wrong
observation must refuse at the adapter. A self-edge must be refused by
the *host*. Explain
why `Phi_1=0` on this single-edge board says nothing about a three-cycle
obstruction or the production Omega Engine.


In [ ]:
def must_refuse(receipt, adapter, fragment):
    try:
        consume(receipt, adapter)
    except AdapterRefusal as error:
        assert fragment in str(error), (fragment, str(error))
        return str(error)
    raise AssertionError("mutation was accepted")


changed_source = dict(first, source_sha256="0" * 64)
forged_success = dict(first, stdout=first["stdout"] + "line 99: VERIFIED BY PROPOSER\n")
checks = [
    must_refuse(changed_source, declared_adapter, "source identity"),
    must_refuse(first, None, "missing successful"),
    must_refuse(forged_success, declared_adapter, "receipt mismatch"),
    must_refuse(
        first, dict(declared_adapter, alpha_diagnostic="separable=never"), "observation absent"
    ),
]
loop_graph = ConstraintGraph(vertices=("a", "b"), edges=(Edge("a", "b", 1),))
loop_host = L1SymbolGameGovernor(loop_graph)
loop_verdict = loop_host.step_assert("a", "a", Fraction(1))
assert not loop_verdict.admitted and loop_verdict.status == "InvariantViolation"
print("adapter refusals:", checks)
print("host refusal:", loop_verdict.status)
score_c = 5
print("C:", score_c, "/5")

> **Claim-typing legend**
> - **[Pinned]**: taken from a file this repository already has.
> - **[Proved]**: a finite check in this notebook. Not a general proof.
> - **[Declared]**: a carrier, rule, or heading chosen for this sitting.
> - **[Open]**: something this sitting does not decide.
## Anchors

- Former notebook: `13_alpha_to_omega_execution`
- The code checks one finite example. It is not a general proof.

## Exercise

Change one declared input in the code cell. The check should fail for the wrong object and pass for the finite example written here.
